# 🌸 鳶尾花隨機森林分類器 API 教學：實作 `/predict` 品種預測端點

> 對象：修習隨機森林分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 113~143 行的 `predict_api` 函數
>
> 這份 Notebook 會帶你**一步步拆解**預測流程，從載入 `.joblib` 模型檔、資料數值校驗，到呼叫 `predict_proba` 取得機率分佈。

## 🎯 學習目標

- [ ] 看懂 FastAPI 端點的基本結構（裝飾器、請求模型、回應模型）
- [ ] 理解 `IrisInput` 欄位驗證 (`ge=0.1, le=10.0`) 防止異常數值輸入
- [ ] 掌握 `model.predict()` 與 `model.predict_proba()` 的回傳結構與型別轉換
- [ ] 將數字類別 ID (0, 1, 2) 映射至花朵品種名稱 (`setosa`, `versicolor`, `virginica`)
- [ ] 能用 `TestClient` 呼叫真實 API 端點並檢查回應與邊界值驗證

## 📌 背景：整個預測流程

訓練階段已經把「隨機森林模型」和「品種標籤名稱 (`target_names`)」存進 `iris_model.joblib`：

```
模型檔 (iris_model.joblib)
├── model        : 訓練好的 RandomForestClassifier 模型
├── target_names : ['setosa', 'versicolor', 'virginica']
└── feature_names: ['sepal length', 'sepal width', 'petal length', 'petal width']
```

預測時，`/predict` 收到使用者的輸入特徵後，執行以下步驟：

```
4項花朵特徵 [sepal_length, sepal_width, petal_length, petal_width]
       │
       ▼
1. model.predict(features)       ──> 取得品種 ID (例如 0)
2. target_names[pred_id]        ──> 取得品種名稱 (例如 'setosa')
3. model.predict_proba(features) ──> 計算各品種信心度與機率字典
       │
       ▼
回傳 IrisOutput 結構化 JSON
```

接下來照這個流程動手做。

In [ ]:
# ============================================
# 0. 載入套件與環境設定
# ============================================

import os, sys, joblib
from pprint import pprint

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print("環境準備完畢。")

---
## Part 1：載入 `.joblib` 模型檔與檢視內容

載入儲存好的模型檔並確認內部物件結構。

In [ ]:
model_path = os.path.join(current_dir, "iris_model.joblib")
if not os.path.exists(model_path):
    from train_save import train_and_save_model
    train_and_save_model()

model_data = joblib.load(model_path)
print("模型檔包含之 Key：", list(model_data.keys()))
print("品種標籤列表：", model_data["target_names"])
print("特徵名稱列表：", model_data["feature_names"])

---
## Part 2：定義 Pydantic 請求與回應模型 (`IrisInput` & `IrisOutput`)

使用 Pydantic 設定輸入範例與數值限制（如長寬需在 0.1 ~ 10.0 cm 之間）。

In [ ]:
from pydantic import BaseModel, Field

class IrisInput(BaseModel):
    sepal_length: float = Field(..., description="花萼長度 (cm)", ge=0.1, le=10.0)
    sepal_width: float = Field(..., description="花萼寬度 (cm)", ge=0.1, le=10.0)
    petal_length: float = Field(..., description="花瓣長度 (cm)", ge=0.1, le=10.0)
    petal_width: float = Field(..., description="花瓣寬度 (cm)", ge=0.1, le=10.0)

class IrisOutput(BaseModel):
    prediction_id: int = Field(..., description="預測類別 ID")
    prediction_label: str = Field(..., description="預測類別名稱")
    probabilities: dict[str, float] = Field(..., description="各類別預測機率")

print("Pydantic 預測模型定義完成！")

---
## Part 3：單獨測試推理邏輯 (`predict` & `predict_proba`)

輸入一組測試資料 `[5.1, 3.5, 1.4, 0.2]`（典型 Setosa 特徵），測試模型輸出。

In [ ]:
model = model_data["model"]
target_names = model_data["target_names"]

sample_features = [[5.1, 3.5, 1.4, 0.2]]

# 1. 預測類別 ID
pred_id = int(model.predict(sample_features)[0])
pred_label = target_names[pred_id]

# 2. 計算各類別預測機率
probs = model.predict_proba(sample_features)[0]
prob_dict = {target_names[i]: float(p) for i, p in enumerate(probs)}

print(f"預測品種 ID: {pred_id}")
print(f"預測品種名稱: {pred_label}")
print("預測機率分佈：")
pprint(prob_dict)

---
## Part 4：建立 FastAPI 應用與 `/predict` 端點

整合模型狀態與 FastAPI 端點。

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Iris Prediction API")

@app.post("/predict", response_model=IrisOutput)
def predict_api(payload: IrisInput):
    features = [[payload.sepal_length, payload.sepal_width, payload.petal_length, payload.petal_width]]
    try:
        pred_id = int(model.predict(features)[0])
        pred_label = target_names[pred_id]
        probs = model.predict_proba(features)[0]
        prob_dict = {target_names[i]: float(p) for i, p in enumerate(probs)}
        return IrisOutput(
            prediction_id=pred_id,
            prediction_label=pred_label,
            probabilities=prob_dict
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"預測失敗: {str(e)}")

print("FastAPI /predict 端點建立成功！")

---
## Part 5：使用 `TestClient` 驗證 `/predict` 端點

測試正常請求與不合法的邊界值請求。

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 正常測試資料
valid_payload = {
    "sepal_length": 6.3,
    "sepal_width": 2.9,
    "petal_length": 5.6,
    "petal_width": 1.8
}
res = client.post("/predict", json=valid_payload)
print("正常請求 HTTP 狀態碼：", res.status_code)
print("預測結果：")
pprint(res.json())

print("\n" + "="*40 + "\n")

# 異常測試資料 (花萼長度為負值 -1.0)
invalid_payload = {
    "sepal_length": -1.0,
    "sepal_width": 3.0,
    "petal_length": 1.5,
    "petal_width": 0.2
}
res_invalid = client.post("/predict", json=invalid_payload)
print("異常請求 HTTP 狀態碼 (應為 422 驗證錯誤)：", res_invalid.status_code)